# COD RTO Risk — honest metrics

Track 02 (AI Risk Manager). Invoice-level RTO **at order confirmation**.

**Honesty contract:** UCI Online Retail II is a UK giftware wholesaler (2009–11). Cancellations are real and **not** India COD RTO.

- **Eval A** — `y_real`: matched full-package cancel (real). Untagged metrics.
- **Eval B** — `y_rto`: documented India/COD simulator. Every metric tagged **`[SCENARIO]`**.

Defense-only: verify or require prepaid — never silent deny. See `ASSUMPTIONS.md`.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
plt.rcParams["figure.figsize"] = (6.2, 3.8)

from src.data import load_orders
from src.overlay import add_overlay_attributes, calibrate_and_label, strip_unobserved
from src.features import assign_split, build_features

orders, data_stats = load_orders()
print("source:", data_stats.get("source"))
print(json.dumps(data_stats, indent=2, default=str)[:1800])
orders.head()

## EDA — prevalence and seasonality (`y_real`)

Guest checkouts are dropped (no history). Invoice-level full-package cancel rate should be **a few percent**, not 28–35%.

In [ ]:
df = orders.copy()
df["month"] = pd.to_datetime(df["invoice_date"]).dt.to_period("M").astype(str)
print("invoices", len(df), "customers", df["customer_id"].nunique())
print("y_real rate", float(df["y_real"].mean()))
print("partial-return rate", float(df["is_partial_return"].mean()))
print("guest line drop rate", data_stats.get("clean", {}).get("guest_line_drop_rate"))

season = df.groupby("month")["y_real"].mean()
ax = season.plot(title="Eval A — monthly full-package cancel rate (y_real)")
ax.set_ylabel("rate")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

df["amount_gbp"].clip(upper=df["amount_gbp"].quantile(0.99)).hist(bins=40)
plt.title("Invoice value (GBP) — UK wholesale, not INR AOV")
plt.show()

## Labels — `y_real` match rule

Greedy most-recent prior `(Customer ID, StockCode)` within 180 days; allocate qty.
`y_real = 1` iff `matched_credited_value / invoice_value >= 0.80`.
Unmatched C-invoices are **not** positives on phantom orders.

In [ ]:
print(df["return_value_frac"].describe())
print("positives", int(df["y_real"].sum()), "partials", int(df["is_partial_return"].sum()))
print("match stats", data_stats.get("match"))

## Overlay + `[SCENARIO]` generator

Logit includes **unobserved** `0.4*z_customer + 0.8*eps` (stripped before modeling). COD/PIN assigned without reading `y_real`.
UK Nov–Dec is a **festive proxy**, not Diwali ground truth.

In [ ]:
ov = add_overlay_attributes(orders)
ov, overlay_meta = calibrate_and_label(ov)
print(overlay_meta["formula"])
print("b0, b_cod", overlay_meta["b0"], overlay_meta["b_cod"])
print("[SCENARIO] realized rates", overlay_meta["realized_rates"])
assert set(overlay_meta["unobserved"]) == {"z_customer", "eps"}
ov = strip_unobserved(ov)
assert "z_customer" not in ov.columns

## Features + temporal split

As-of `InvoiceDate` only. PIN/category rates fit on **train** (Mar 2010–Aug 2011), mapped to val/test.
No random k-fold — festive leakage.

In [ ]:
ov["split"] = assign_split(ov["invoice_date"])
print(ov.groupby("split").size())
print("y_real by split\n", ov.groupby("split")["y_real"].mean())
print("[SCENARIO] y_rto by split\n", ov.groupby("split")["y_rto"].mean())

df_a, spec_a = build_features(ov, y_col="y_real", eval_name="A")
df_b, spec_b = build_features(ov, y_col="y_rto", eval_name="B")
print("Eval A features", spec_a["num_cols"] + spec_a["cat_cols"])
print("Eval B features", spec_b["num_cols"] + spec_b["cat_cols"])
print("train PIN rates [SCENARIO]", spec_b.get("pin_rto_rate"))

## Baseline logistic + XGBoost, PR, calibration, cost

Training lives in `src.pipeline`. This cell runs it if `artifacts/metrics.json` is missing, otherwise loads artifacts (so the notebook stays the evidence trail without double-fitting unless needed).

Operating point = **cost-optimal** `t*`, not 0.5, not max-F1. Headline curve is **PR**.

In [ ]:
metrics_path = ROOT / "artifacts" / "metrics.json"
if not metrics_path.exists():
    from src.pipeline import run_pipeline
    metrics = run_pipeline()
else:
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))

print("source", metrics.get("source"))
print("[SCENARIO] realized", metrics.get("realized_scenario_rates"))
print("Eval A", {k: metrics["eval_a"][k] for k in ("precision", "recall", "f1", "average_precision", "threshold", "ece")})
print("[SCENARIO] Eval B", {k: metrics["eval_b"][k] for k in ("precision", "recall", "f1", "average_precision", "threshold", "ece", "tag")})
print("logistic top (B)", metrics["eval_b"].get("logistic_top_coefs", [])[:6])
print("fairness", metrics["eval_b"].get("fairness_note"))

In [ ]:
from IPython.display import Image, Markdown, display

art = ROOT / "artifacts"
display(Markdown("### Eval A — PR (real cancellations)"))
display(Image(filename=str(art / "pr_curve_eval_a.png")))
display(Markdown("### Eval A — calibration"))
display(Image(filename=str(art / "calibration_eval_a.png")))
display(Markdown("### `[SCENARIO]` Eval B — PR"))
display(Image(filename=str(art / "pr_curve_eval_b.png")))
display(Markdown("### `[SCENARIO]` Eval B — calibration"))
display(Image(filename=str(art / "calibration_eval_b.png")))
display(Markdown("### `[SCENARIO]` cost vs threshold + FP-cost sensitivity"))
display(Image(filename=str(art / "cost_vs_threshold.png")))
display(Image(filename=str(art / "fp_cost_sensitivity.png")))

## Policy (defense-only)

- Low (`p < t_med`): proceed, COD allowed
- Medium: confirmation call / SMS / WhatsApp **before dispatch**
- High (`p ≥ t*`): require prepaid **or** address re-verification — **never `denied`**

Unknown customer → medium + `insufficient_history`.

PIN-tier FP rates are in `metrics.json` → `eval_b.fp_rate_by_pincode_tier`. Extra verification, never “we don’t ship to this PIN.”